# Phase 2 — Pretraining Model H (Hindi) and Model L (Nepali)

Language Models and Agents, Monsoon 2026 — Individual Project.

Two independent ~25M-parameter decoder-only Transformers, trained from
scratch on their own corpora with their own tokenizers. No pretrained
weights, no shared vocabulary, no HuggingFace model classes.

**Before running, in the sidebar:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (see below) |
| Persistence | **Files only** |
| Input | `lma-code`, `lma-data-hindi`, `lma-data-nepali` |

T4 x2 over P100: T4 has tensor cores (~65 TFLOPS fp16), P100 does not
(~19 TFLOPS). This training is AMP fp16, so tensor cores matter — one T4
already beats a P100 ~3x, and the trainer uses BOTH cards via DataParallel.

**Set `LANGUAGE` in Cell 2**, then Run All. One language per session —
each takes ~5-6 hours and a session caps at 12.


## Cell 1 — Environment

Kaggle mounts datasets read-only, so the code is copied somewhere writable
and the corpora are **symlinked** rather than copied: they are 5.5 GB
uncompressed against a 20 GB `/kaggle/working` budget that the checkpoints
also need.


In [ ]:
import os, shutil, subprocess, sys, time

REPO = '/kaggle/working/LMA_IDV_Proj'
MARKER = os.path.join('shared', 'model', 'transformer.py')


def find_dir_containing(marker, roots=('/kaggle/input',), max_depth=6):
    """Find the directory holding *marker*, wherever Kaggle mounted it.

    Mount layout differs by how the notebook was created: the UI gives a
    flat /kaggle/input/<slug>/, while an API-pushed notebook nests them
    under /kaggle/input/datasets/<owner>/<slug>/. Searching for a marker
    file works under both instead of hardcoding either.
    """
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _files in os.walk(root):
            if dirpath.count(os.sep) - root.count(os.sep) > max_depth:
                dirnames[:] = []
                continue
            if os.path.exists(os.path.join(dirpath, marker)):
                return dirpath
    return None


src_root = find_dir_containing(MARKER)
if src_root is None:
    print('Could not find the code dataset. What IS mounted:')
    for dirpath, dirnames, files in os.walk('/kaggle/input'):
        if dirpath.count(os.sep) <= 5:
            print(' ', dirpath, sorted(dirnames)[:6], sorted(files)[:4])
    raise FileNotFoundError('attach the lma-code dataset to this notebook')

print('code   :', src_root)
if os.path.exists(REPO):
    shutil.rmtree(REPO)
shutil.copytree(src_root, REPO)
sys.path.insert(0, REPO)
os.chdir(REPO)
print('repo   :', REPO)


In [ ]:
def link_corpus(lang):
    """Symlink one language's tokenized .bin files AND held-out .txt splits
    into the paths its config expects.

    Symlinks rather than copies: the corpora are large and /kaggle/working
    has a 20 GB budget the checkpoints also draw on. .txt splits are needed
    only for BPB's UTF-8 byte count (validation.txt / test.txt) -- without
    them, evaluate_lm.py silently falls back to byte counts recorded in the
    YAML config instead of measuring the mounted corpus.
    """
    def link_dir(src, dst):
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            link = os.path.join(dst, f)
            if os.path.islink(link) or os.path.exists(link):
                os.remove(link) if os.path.islink(link) else None
            if not os.path.exists(link):
                os.symlink(os.path.join(src, f), link)

    bin_src = None
    txt_src = None
    for dirpath, _dirs, files in os.walk('/kaggle/input'):
        norm = dirpath.replace(os.sep, '/')
        if 'train.bin' in files and f'{lang}/data/tokenized' in norm:
            bin_src = dirpath
        if 'validation.txt' in files and f'{lang}/data/final' in norm:
            txt_src = dirpath
    if bin_src is None:
        print(f'  {lang}: not mounted — skipping')
        return False

    link_dir(bin_src, f'{REPO}/{lang}/data/tokenized')
    n = os.path.getsize(os.path.join(bin_src, 'train.bin')) // 4   # uint32
    print(f'  {lang}: {n:,} train tokens', end='')

    if txt_src is not None:
        link_dir(txt_src, f'{REPO}/{lang}/data/final')
        print(' | validation.txt + test.txt linked (BPB will use real bytes)')
    else:
        print(' | .txt splits not mounted — BPB falls back to config byte_counts')
    return True


print('corpora:')
for lang in ('hindi', 'nepali'):
    link_corpus(lang)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'sacrebleu', 'rouge-score'], check=False)

# Kaggle's font set differs from a local Linux box, and matplotlib's
# default has no Devanagari glyphs — it drops them with only a warning,
# which would silently ruin every attention heatmap.
subprocess.run('apt-get install -qq -y fonts-lohit-deva fonts-indic',
               shell=True, check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import torch
print('torch :', torch.__version__)
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    p = torch.cuda.get_device_properties(0)
    print(f'GPU   : {n} x {p.name}  ({p.total_memory / 1e9:.1f} GB each)')
else:
    print('GPU   : NONE — set Accelerator to GPU T4 x2 in the sidebar')


## Cell 2 — Choose the language

One language per session. Set this, then Run All.


In [ ]:
LANGUAGE = 'hindi'          # 'hindi' (Model H) or 'nepali' (Model L)

CONFIG = f'{LANGUAGE}/configs/{LANGUAGE}_model.yaml'
CKPT_DIR = f'{LANGUAGE}/checkpoints'
BEST = f'{CKPT_DIR}/best.pt'
LATEST = f'{CKPT_DIR}/latest.pt'

import yaml
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

print(f"{cfg['model_label']}")
print(f"  steps      : {cfg['training']['max_steps']:,}")
print(f"  tokens/step: {cfg['training']['micro_batch_size'] * cfg['model']['block_size'] * cfg['training']['grad_accum_steps']:,}")
print(f"  vocab      : {cfg['model']['vocab_size']:,}")


## Cell 3 — Correctness gates

**Run this before training, every session.** It takes under a minute and
catches the bugs that would otherwise waste six hours of GPU time:
an off-by-one causal mask or a mask applied after softmax both produce a
model that appears to train while having leaked the future.

Expect `22 passed`, `Target 25,321,472 — CONFIRMED`, and `PASSED` with a
max delta of exactly `0.0`.


In [ ]:
gate = subprocess.run([sys.executable, '-m', 'pytest',
                       'tests/test_phase2.py', '-q'],
                      capture_output=True, text=True)
print(gate.stdout[-1500:])
assert gate.returncode == 0, 'correctness gates FAILED — do not train'


In [ ]:
!python {LANGUAGE}/train/count_params.py
!python {LANGUAGE}/eval/verify_causal_masking.py


## Cell 4 — Pretrain

Runs one full epoch. Resumes automatically from `latest.pt` if a previous
session was interrupted — re-running this identical cell is the recovery
procedure.

**Do not pass `--max-steps` to finish faster.** `max_steps` defines the
cosine schedule, so changing it changes the learning rate at every
remaining step and the run is no longer the one you started.

If it dies with CUDA OOM, use the fallback in the next cell instead:
a smaller micro-batch with more accumulation keeps the effective batch
at 49,152 tokens.


In [ ]:
t0 = time.time()
!python {LANGUAGE}/train/pretrain.py
print(f'\nwall clock: {(time.time() - t0) / 3600:.2f} h')


In [ ]:
# OOM fallback — same effective batch (16 x 512 x 6 = 49,152 tokens/step).
# Uncomment and run INSTEAD of the cell above.
# !python {LANGUAGE}/train/pretrain.py --micro-batch-size 16 --grad-accum-steps 6


## Cell 5 — Back up the checkpoint

**Run this every session.** `/kaggle/working` survives between runs of the
same notebook with Persistence on, but it is not a backup. Download the
zip from the Output tab, put it on Google Drive, and paste the link into
the README — the spec requires Drive links for checkpoints.


In [ ]:
stamp = time.strftime('%Y%m%d_%H%M')
out = f'/kaggle/working/{LANGUAGE}_phase2_{stamp}.zip'
!cd {REPO} && zip -q -1 -r {out} {LANGUAGE}/checkpoints {LANGUAGE}/logs
print(out, f'{os.path.getsize(out) / 1e6:.0f} MB')
print('-> download from the Output tab, then upload to Drive')


## Cell 6 — Evaluate

Produces every Phase 2 deliverable: CE/PPL/BPB, generation at greedy and
T=0.5/1.0/1.5 scored with BLEU-4/chrF/ROUGE-L/Distinct-1,2/repetition, and
attention entropy, mean distance and heatmaps.

Uses `best.pt` (lowest validation loss), not `latest.pt`.


In [ ]:
ckpt = BEST if os.path.exists(BEST) else LATEST
assert os.path.exists(ckpt), 'no checkpoint — run Cell 4 first'
print('evaluating', ckpt)

!python {LANGUAGE}/eval/evaluate_lm.py --checkpoint {ckpt} --splits validation test
!python {LANGUAGE}/eval/evaluate_generation.py --checkpoint {ckpt}
!python {LANGUAGE}/eval/analyze_attention.py --checkpoint {ckpt} --device cpu


In [ ]:
# Confirm Devanagari actually rendered. matplotlib only WARNS about
# missing glyphs, so a heatmap of empty boxes otherwise looks like a
# success. Check this before trusting the figures.
from shared.eval import plots
font = plots.configure_devanagari_font()
ok = plots.verify_devanagari_rendering()
print(f'font: {font}  renders Devanagari: {ok}')
if not ok:
    print('WARNING: re-run the apt-get in Cell 1, then this cell')


## Cell 7 — Collect the graded artifacts

The spec is explicit: *"Graders will not open external dashboards or Drive
folders to find figures; if it is not in the branch, it is not graded."*

Download this zip and **commit its contents to the `phase-2` branch** —
figures, metric JSON, and training logs all belong in git. Only the
checkpoints stay on Drive.


In [ ]:
art = f'/kaggle/working/{LANGUAGE}_artifacts_{stamp}.zip'
targets = f'report/figures {LANGUAGE}/eval/results {LANGUAGE}/logs'
!cd {REPO} && zip -q -r {art} {targets} -x '*.pt' '*.gitkeep'
print(art, f'{os.path.getsize(art) / 1e6:.1f} MB')
!cd {REPO} && unzip -l {art} | tail -25


## Cell 8 — Summary

Both languages must be trained before the report has anything to compare.
Run this notebook once with `LANGUAGE = 'hindi'`, then again with
`LANGUAGE = 'nepali'`, then run `report/scripts/build_phase2_report.py`
locally once both sets of results are committed.


In [ ]:
import json

res = f'{LANGUAGE}/eval/results/lm_metrics.json'
if os.path.exists(res):
    with open(res) as f:
        m = json.load(f)['splits']['validation']
    print(f"{cfg['model_label']}")
    print(f"  CE loss    : {m['cross_entropy_nats']:.4f} nats/token")
    print(f"  perplexity : {m['perplexity']:.2f}")
    print(f"  BPB        : {m.get('bits_per_byte')}")
else:
    print('no LM metrics yet — run Cell 6')

other = 'nepali' if LANGUAGE == 'hindi' else 'hindi'
print(f'\nNext: set LANGUAGE = {other!r} and Run All in a new session.')
